# Pipeline CRM Atlas — Glue / Spark

Versión Spark del pipeline CRM. Reproduce exactamente la misma lógica que `PipelineCrm_enhanced.ipynb` usando exclusivamente fuentes S3 (Silver DL). Escribe los reportes en Gold S3.

In [ ]:
%idle_timeout 720
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5
%extra_py_files s3://aws-s3-reporte-atenea-usorg-nprd/glue-libs/


In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from datetime import datetime
import boto3

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
spark.conf.set('spark.sql.session.timeZone', 'America/Mexico_City')
print('Spark session iniciada.')


In [ ]:
BUCKET      = 'aws-s3-reporte-atenea-usorg-nprd'
BASE        = f's3://{BUCKET}/data/'
GOLD_BASE   = f's3://{BUCKET}/data/Gold/Drive/'
FECHA_CORTE = '2026-07-01'

# Rutas relativas Silver DL
SOURCES = {
    'historico_oportunidades': 'Silver/Atlas/ProcHistoricoOportunidades/',
    'citas':                   'Silver/Atlas/ProcCitas/',
    'oportunidades':           'Silver/Atlas/ProcOportunidades/',
    'historico_citas':         'Silver/Atlas/ProcHistoricoCitas/',
    'historico_casos':         'Silver/Atlas/ProcCasosCrmHistorico/',
    'casos':                   'Silver/Atlas/ProcCasosCrm/',
    'pedidos':                 'Silver/Atlas/ProcPedidos/',
    'solicitudes_credito':     'Silver/Atlas/ProcSolicitudesCredito/',
    'catalogo_usuarios':       'Silver/Atlas/ProcCatalogoUsuarios/',
    'clientes':                'Silver/Atlas/ProcClientes/',
}

# ----- Reglas de negocio (desde crm_config) -----
ID_AM_DUMMIES = [1067687, 1067696, 1067711, 1067720,
                 1067648, 1067654, 1067669, 1067678]

STATUS_PEDIDOS_ABIERTOS = [
    'revision de auto', 'auto en cita', 'negociacion de precio',
    'acuerdo de compraventa', 'formalizacion de compra', 'cita para entrega',
]

WORK_TYPES_COMPRADOR = ['cita inicial visita comprador', 'cita final busca vehiculo']
WORK_TYPES_VENDEDOR  = [
    'cita recogida vehiculo', 'cita final entrega vehiculo',
    'revision mecanica inicial', 'revision mecanica final',
    'cita inicial entrega vehiculo', 'recogida vehiculo',
]

STATUS_SHOW_CITAS    = ['completa', 'en progreso']
STATUS_HISTSHOW      = ['completed', 'in progress']
STATUS_CANCELADA     = [
    'canceled', 'canceledbuyer', 'canceledseller',
    'noshowbuyer', 'noshowseller', 'no show - buyer',
]

EQUIPOS_ESPACIOS     = ['patriotismo', 'satelite', 'torre_bbva', 'gran sur', 'torre bbva']
EQUIPOS_SC           = ['cdmx', 'mid', 'celula credito']

# ----- Columnas de salida (desde crm_config) -----
REPORTE_OPORTUNIDADES_COLS = [
    'opportunity_id','opportunity_name','opportunity_created_date',
    'opportunity_created_date_day','fecha_asignacion','opportunity_owner','owner_id',
    'opportunity_owner_equipo','nombre_comprador','id_am_comprador',
    'last_modified_date','opportunity_source','opportunity_source_aux','opportunity_stage',
    'flag_caso_tomado_perf_sc','asesor_caso_tomado_perf_sc','equipo_asesor_caso_tomado_perf_sc',
    'fecha_asignacion_perfilamiento_sc','fecha_caso_tomado_sc',
    'case_owner_name_perf_sc','case_owner_id_perf_sc','case_owner_equipo_perf_sc',
    'case_id_perfilamiento_sc','case_status_perfilamiento_sc',
    'perf_bc_score','perf_intencion_pago','perf_contactado','perf_interesado',
    'perf_comentarios','perf_canal_respuesta','perf_fecha_primer_contacto',
    'perf_fecha_ultimo_contacto','perf_contacto_otro_momento','perf_fecha_sig_contacto',
    'flag_perfilamento_credito','asesor_perfilamiento_credito',
    'asesor_perfilamiento_credito_id','equipo_asesor_perfilamiento_credito',
    'fecha_asignacion_perfilamiento_credito',
    'case_id_perfilamiento_credito','case_status_perfilamiento_credito',
    'cred_contactado','cred_perfilamiento_credit','cred_canal_respuesta_credito',
    'numero_pedidos','numero_pedidos_abiertos','fecha_primer_pedido','fecha_ultimo_pedido',
    'n_simulaciones_credito','n_simulaciones_con_folio',
    'n_simulaciones_preaceptadas','n_simulaciones_contingencia',
    'simulation_name_op','created_date_sim_op','tipo_credito',
    'flag_cita_agendada_oportunidad','flag_cita_sin_pedido_agendada',
    'flag_cita_sin_pedido_show','fecha_primera_cita_visita_comp',
    'fecha_ultima_cita_visita_comp','flag_cita_show_oportunidad',
    'stage_antes_de_cierre','usuario_que_cerro','equipo_que_cerro','fecha_cierre',
    'motivo_cierre_oportunidad','detalle_motivo_cierre_oportunidad',
    'primer_booker_comprador','equipo_primer_booker_comprador',
]

REPORTE_CITAS_COLS = [
    'numero_cita','created_time','created_date','status','sf_order_id','rol',
    'work_type_name','id_am','nombre','telefono','email','espacio_cita',
    'sched_date','sched_start_time','sched_end_time',
    'opportunity_id','opportunity_name','opportunity_created_date',
    'opportunity_owner_id','opportunity_owner','opportunity_owner_equipo',
    'opportunity_source','opportunity_source_aux','opportunity_stage',
    'fecha_asignacion','perf_intencion_pago','perf_bc_score',
    'perf_sc_case_owner_id','perf_sc_case_owner_name',
    'booker_name','booker_equipo','booker_equipo_operativo',
    'status_cancelacion','motivo_cancelacion','cancelado_por_id',
    'cancelado_por','cancelado_por_equipo','cancelado_por_equipo_operativo',
    'check_in','check_out','fecha_reagendamiento',
    'flag_dummy','flag_duplicada','flag_cita_agendada_comprador',
    'flag_cita_show_comprador','flag_booker_origen_nulo','flag_oppid_origen_nulo',
]


## 1. Lectura de fuentes desde Silver DL

In [ ]:
def latest_partition(base_s3_path):
    s3 = boto3.client('s3')
    bucket, prefix = base_s3_path.replace('s3://', '').split('/', 1)
    prefix = prefix.rstrip('/') + '/'
    def max_partition(pfx, key):
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=pfx, Delimiter='/')
        parts = [(int(p['Prefix'].rstrip('/').split('=')[-1]), p['Prefix'])
                 for p in resp.get('CommonPrefixes', []) if '=' in p['Prefix']]
        if not parts: raise FileNotFoundError(f'No partitions ({key}=) under {pfx}')
        return max(parts)[1]
    year_pfx  = max_partition(prefix,   'year')
    month_pfx = max_partition(year_pfx, 'month')
    day_pfx   = max_partition(month_pfx, 'day')
    full = f's3://{bucket}/{day_pfx}'
    print(f'  -> {full}')
    return full


In [ ]:
_t0 = datetime.now()
_dfs = {}
for nombre, path in SOURCES.items():
    print(f'Leyendo {nombre}...')
    _dfs[nombre] = spark.read.parquet(latest_partition(BASE + path))
    print(f'  -> {_dfs[nombre].count():,} filas')

historico_oportunidades = _dfs['historico_oportunidades']
citas                   = _dfs['citas']
oportunidades           = _dfs['oportunidades']
historico_citas         = _dfs['historico_citas']
historico_casos         = _dfs['historico_casos']
casos                   = _dfs['casos']
pedidos                 = _dfs['pedidos']
solicitudes_credito     = _dfs['solicitudes_credito']
catalogo_usuarios       = _dfs['catalogo_usuarios']
clientes                = _dfs['clientes']
print(f'Listo en {(datetime.now()-_t0).seconds}s')


In [ ]:
def filtrar_fecha(df, col, corte=FECHA_CORTE):
    if col not in df.columns:
        print(f'  [SKIP] {col!r} no encontrada')
        return df
    return df.filter(F.col(col) > F.lit(corte))

historico_oportunidades = filtrar_fecha(historico_oportunidades, 'created_date')
citas                   = filtrar_fecha(citas,                   'created_date')
oportunidades           = filtrar_fecha(oportunidades,           'opportunity_created_date')
historico_citas         = filtrar_fecha(historico_citas,         'created_date')
historico_casos         = filtrar_fecha(historico_casos,         'case_created_date')
casos                   = filtrar_fecha(casos,                   'case_created_date')


## 2. Helpers: catálogo SCD2 + función add_team_scd2

In [ ]:
# Si el catalogo no trae periodos SCD2 se agregan como siempre-vigentes
if 'start_date' not in catalogo_usuarios.columns:
    catalogo_usuarios = (
        catalogo_usuarios
        .withColumn('start_date', F.lit('2000-01-01').cast('date'))
        .withColumn('end_date',   F.lit('2099-12-31').cast('date'))
    )

# Una fila por usuario (la mas reciente por processing_date) para evitar fan-out
_w_cat = Window.partitionBy('id').orderBy(F.col('processing_date').desc())
_cat_dedup = (
    catalogo_usuarios
    .withColumn('_rn', F.row_number().over(_w_cat))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
)

def add_team_scd2(df, date_col, user_id_col, output_col):
    """Une el df con el catalogo deduplicado para obtener el equipo vigente en date_col."""
    p = f'_scd_{output_col}'
    _cat = _cat_dedup.select(
        F.col('id').alias(f'{p}_id'),
        F.col('equipo').alias(output_col),
        F.col('start_date').alias(f'{p}_s'),
        F.col('end_date').alias(f'{p}_e'),
    )
    return (
        df.join(_cat, on=(
            (F.col(user_id_col).cast('string') == F.col(f'{p}_id').cast('string')) &
            (F.col(date_col).cast('date') >= F.col(f'{p}_s')) &
            (F.col(date_col).cast('date') <= F.col(f'{p}_e'))
        ), how='left')
        .drop(f'{p}_id', f'{p}_s', f'{p}_e')
    )


## 3. Reporte Oportunidades

In [ ]:
# ---- Casos de perfilamiento Sales Center ----
# case_subject ya viene en minusculas desde Silver DL, pero aplicamos lower() por seguridad
_casos_sc = (
    casos
    .filter(F.lower(F.trim(F.col('case_subject'))) == 'perfilamiento contact center')
    .select(
        'opportunity_id',
        F.col('case_id').alias('case_id_perfilamiento_sc'),
        F.col('case_status').alias('case_status_perfilamiento_sc'),
        'case_owner_name_perf_sc',
        'case_owner_id_perf_sc',
    )
)

# ---- Casos de perfilamiento Credito ----
_casos_cred = (
    casos
    .filter(F.lower(F.trim(F.col('case_subject'))) == 'perfilamiento credito')
    .select(
        'opportunity_id',
        F.col('case_id').alias('case_id_perfilamiento_credito'),
        F.col('case_status').alias('case_status_perfilamiento_credito'),
    )
)
print(f'casos SC: {_casos_sc.count():,} | casos credito: {_casos_cred.count():,}')


In [ ]:
# ---- Asesor SC: quien tomo el caso (abierto -> in progress) ----
_w_sc = Window.partitionBy('case_id').orderBy('created_date')
_asesor_sc = (
    historico_casos
    .filter(F.col('case_id').isin(
        [r.case_id_perfilamiento_sc for r in _casos_sc.select('case_id_perfilamiento_sc').collect()]
    ))
    .filter(F.lower(F.trim(F.col('field')))     == 'status')
    .filter(F.lower(F.trim(F.col('old_value'))) == 'abierto')
    .filter(F.lower(F.trim(F.col('new_value'))) == 'in progress')
    .withColumn('_rn', F.row_number().over(_w_sc))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('case_id').alias('case_id_perfilamiento_sc'),
        F.col('created_date').alias('fecha_caso_tomado_sc'),
        F.upper(F.col('created_by')).alias('asesor_caso_tomado_perf_sc'),
        F.col('created_by_id').alias('asesor_perfilamiento_sc_id'),
    )
)

# ---- Fecha de asignacion SC: primer cambio de owner desde 'caso de servicio' ----
_w_asig = Window.partitionBy('case_id').orderBy('created_date')
_fecha_asig_sc = (
    historico_casos
    .filter(F.col('case_id').isin(
        [r.case_id_perfilamiento_sc for r in _casos_sc.select('case_id_perfilamiento_sc').collect()]
    ))
    .filter(F.lower(F.trim(F.col('field')))     == 'owner')
    .filter(F.lower(F.trim(F.col('old_value'))) == 'caso de servicio')
    .filter(F.lower(F.trim(F.col('new_value'))) != 'caso de servicio')
    .withColumn('_rn', F.row_number().over(_w_asig))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('case_id').alias('case_id_perfilamiento_sc'),
        F.col('created_date').alias('fecha_asignacion_perfilamiento_sc'),
    )
)

# ---- Asesor Credito: quien tomo el caso de credito (abierto -> in progress) ----
_w_cred = Window.partitionBy('case_id').orderBy('created_date')
_asesor_cred = (
    historico_casos
    .filter(F.col('case_id').isin(
        [r.case_id_perfilamiento_credito for r in _casos_cred.select('case_id_perfilamiento_credito').collect()]
    ))
    .filter(F.lower(F.trim(F.col('field')))     == 'status')
    .filter(F.lower(F.trim(F.col('old_value'))) == 'abierto')
    .filter(F.lower(F.trim(F.col('new_value'))) == 'in progress')
    .withColumn('_rn', F.row_number().over(_w_cred))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('case_id').alias('case_id_perfilamiento_credito'),
        F.col('created_date').alias('fecha_asignacion_perfilamiento_credito'),
        F.upper(F.col('created_by')).alias('asesor_perfilamiento_credito'),
        F.col('created_by_id').alias('asesor_perfilamiento_credito_id'),
    )
)
print('Historico casos procesado.')


In [ ]:
# Una fila por opportunity: conteos y fechas de pedidos
_summary_ped = (
    pedidos
    .withColumn('_pedido_abierto',
        F.when(F.lower(F.col('sf_order_status')).isin(STATUS_PEDIDOS_ABIERTOS), 1).otherwise(0))
    .groupBy('opportunity_id')
    .agg(
        F.countDistinct('sf_order_id').alias('numero_pedidos'),
        F.sum('_pedido_abierto').alias('numero_pedidos_abiertos'),
        F.min('fecha_de_creacion').alias('fecha_primer_pedido'),
        F.max('fecha_de_creacion').alias('fecha_ultimo_pedido'),
    )
)
print(f'summary pedidos: {_summary_ped.count():,} oportunidades con pedido')


In [ ]:
# Una fila por opportunity: conteos de simulaciones y flags de credito
_summary_sol = (
    solicitudes_credito
    .withColumn('_folio_num', F.col('folio').cast('double'))
    .withColumn('_flag_folio',
        F.when(F.col('_folio_num').isNotNull() & (F.col('_folio_num') > 0), 1).otherwise(0))
    .withColumn('_flag_contingencia',
        F.when(F.lower(F.col('tipo_conclusion_flujo_credito')) == 'flujo contingente', 1).otherwise(0))
    .withColumn('_flag_aceptada',
        F.when(
            (F.lower(F.col('status_solicitud')).contains('acept')) & (F.col('_flag_folio') == 1),
            1).otherwise(0))
    .groupBy('opportunity_id')
    .agg(
        F.countDistinct('simulation_name').alias('n_simulaciones_credito'),
        F.sum('_flag_folio').alias('n_simulaciones_con_folio'),
        F.sum('_flag_aceptada').alias('n_simulaciones_preaceptadas'),
        F.sum('_flag_contingencia').alias('n_simulaciones_contingencia'),
    )
)
print(f'summary simulaciones: {_summary_sol.count():,} oportunidades con simulacion')


In [ ]:
# ---- Citas comprador agrupadas por oportunidad ----
_citas_comp = (
    citas
    .filter(F.col('opportunity_id').isNotNull())
    .filter(
        F.lower(F.coalesce(F.col('work_type_name'), F.lit(''))).isin(
            WORK_TYPES_COMPRADOR + ['']) |
        F.col('work_type_name').isNull()
    )
    .withColumn('_cita_completa',
        F.when(F.lower(F.col('status')).isin(STATUS_SHOW_CITAS), 1).otherwise(0))
    .withColumn('_sched_date', F.date_format(F.col('sched_start_time').cast('date'), 'yyyy-MM-dd'))
)

_summary_citas = (
    _citas_comp
    .groupBy('opportunity_id')
    .agg(
        F.countDistinct('created_date').alias('numero_citas_comprador'),
        F.min('_sched_date').alias('fecha_primera_cita_visita_comp'),
        F.max('_sched_date').alias('fecha_ultima_cita_visita_comp'),
        F.sum('_cita_completa').alias('_citas_completas_sum'),
    )
    .withColumn('flag_cita_agendada_oportunidad',
        F.when(F.col('numero_citas_comprador') > 0, 1).otherwise(0))
    .withColumn('flag_cita_show_oportunidad',
        F.when(F.col('_citas_completas_sum') > 0, 1).otherwise(0))
    .drop('_citas_completas_sum')
)

# ---- Citas proactivas (sin opportunity_id y sin pedido): para flags de oportunidad ----
_citas_proactivas = (
    citas
    .join(pedidos.select('commerce_order_id', 'sf_order_id').distinct(),
          on='commerce_order_id', how='left')
    .filter(F.col('opportunity_id').isNull())
    .filter(
        F.lower(F.coalesce(F.col('work_type_name'), F.lit(''))).isin(
            WORK_TYPES_COMPRADOR + ['']) |
        F.col('work_type_name').isNull()
    )
    .filter(F.col('sf_order_id').isNull())
    .select('id_am', 'status')
    .distinct()
)
_ids_proactivas       = set(r.id_am for r in _citas_proactivas.select('id_am').collect() if r.id_am)
_ids_proactivas_show  = set(
    r.id_am for r in
    _citas_proactivas.filter(F.lower(F.col('status')).isin(STATUS_SHOW_CITAS))
    .select('id_am').collect() if r.id_am
)
print(f'IDs proactivas: {len(_ids_proactivas)} | show: {len(_ids_proactivas_show)}')


In [ ]:
# ---- Cierres de oportunidad: ultimo stage antes de 'cerrada (perdida)' ----
_w_cierre = Window.partitionBy('opportunity_id').orderBy(F.col('created_date').desc())
_hist_opp_mod = (
    historico_oportunidades
    .filter(F.lower(F.trim(F.col('new_value'))) == 'cerrada (perdida)')
    .withColumn('_rn', F.row_number().over(_w_cierre))
    .filter(F.col('_rn') == 1).drop('_rn')
    .withColumnRenamed('old_value',    'stage_antes_de_cierre')
    .withColumnRenamed('created_by',   'usuario_que_cerro')
    .withColumnRenamed('created_by_id','id_usuario_que_cerro')
    .withColumnRenamed('created_date', 'fecha_cierre')
    .select('opportunity_id', 'stage_antes_de_cierre',
            'usuario_que_cerro', 'id_usuario_que_cerro', 'fecha_cierre')
)
# Equipo de quien cerro via SCD2
_hist_opp_mod = add_team_scd2(
    _hist_opp_mod, 'fecha_cierre', 'id_usuario_que_cerro', 'equipo_que_cerro'
).drop('id_usuario_que_cerro')
print(f'cierres historico: {_hist_opp_mod.count():,}')


In [ ]:
# ---- Primer booker por oportunidad: quien agendo la primera cita del comprador ----
# Paso 1: ultima cita comprador por oportunidad (por created_date asc = la primera)
_w_cita_op = Window.partitionBy('opportunity_id').orderBy(F.col('created_date').asc())
_primera_cita = (
    citas
    .filter(F.col('opportunity_id').isNotNull())
    .filter(
        F.lower(F.coalesce(F.col('work_type_name'), F.lit(''))).isin(
            WORK_TYPES_COMPRADOR + ['']) |
        F.col('work_type_name').isNull()
    )
    .withColumn('_rn', F.row_number().over(_w_cita_op))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select('opportunity_id', 'numero_cita')
)

# Paso 2: en historico_citas, quien cambio status null -> 'scheduled' (es el booker)
_w_booker = Window.partitionBy('numero_cita').orderBy('created_date')
_bookers_citas = (
    historico_citas
    .filter(F.lower(F.trim(F.col('field')))     == 'status')
    .filter(F.col('old_value').isNull())
    .filter(F.lower(F.trim(F.col('new_value'))) == 'scheduled')
    .withColumn('_rn', F.row_number().over(_w_booker))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        'numero_cita',
        F.upper(F.col('created_by')).alias('primer_booker_comprador'),
        F.col('created_by_id').alias('_primer_booker_id'),
        F.col('created_date').alias('_primer_booker_fecha'),
    )
)

# Paso 3: unir primer cita por oportunidad con booker, y obtener su equipo
_primer_booker_df = (
    _primera_cita
    .join(_bookers_citas, on='numero_cita', how='left')
)
_primer_booker_df = add_team_scd2(
    _primer_booker_df, '_primer_booker_fecha', '_primer_booker_id', 'equipo_primer_booker_comprador'
).drop('numero_cita', '_primer_booker_id', '_primer_booker_fecha')
print(f'primer booker: {_primer_booker_df.count():,} oportunidades')


In [ ]:
# ---- Ensamble principal de oportunidades_consumo ----
oportunidades_consumo = (
    oportunidades
    # 1. equipo del owner de la oportunidad (SCD2)
    .transform(lambda df: add_team_scd2(
        df, 'opportunity_created_date', 'owner_id', 'opportunity_owner_equipo'))
    # 2. caso de perfilamiento SC
    .join(_casos_sc, on='opportunity_id', how='left')
    # 3. fecha de asignacion al SC
    .join(_fecha_asig_sc, on='case_id_perfilamiento_sc', how='left')
    # 4. equipo del case_owner SC al momento de la asignacion (SCD2)
    .transform(lambda df: add_team_scd2(
        df, 'fecha_asignacion_perfilamiento_sc', 'case_owner_id_perf_sc', 'case_owner_equipo_perf_sc'))
    # 5. caso de perfilamiento credito
    .join(_casos_cred, on='opportunity_id', how='left')
    # 6. asesor que tomo el caso SC
    .join(_asesor_sc, on='case_id_perfilamiento_sc', how='left')
    # 7. equipo del asesor SC (SCD2)
    .transform(lambda df: add_team_scd2(
        df, 'fecha_caso_tomado_sc', 'asesor_perfilamiento_sc_id', 'equipo_asesor_caso_tomado_perf_sc'))
    .drop('asesor_perfilamiento_sc_id')
    # 8. asesor que tomo el caso credito
    .join(_asesor_cred, on='case_id_perfilamiento_credito', how='left')
    # 9. equipo asesor credito (SCD2)
    .transform(lambda df: add_team_scd2(
        df, 'fecha_asignacion_perfilamiento_credito', 'asesor_perfilamiento_credito_id', 'equipo_asesor_perfilamiento_credito'))
    .drop('asesor_perfilamiento_credito_id')
    # 10. resumen de pedidos (1:1)
    .join(_summary_ped, on='opportunity_id', how='left')
    # 11. resumen simulaciones credito (1:1)
    .join(_summary_sol, on='opportunity_id', how='left')
    # 12. resumen citas comprador (1:1)
    .join(_summary_citas, on='opportunity_id', how='left')
    # 13. cierres (cerrada perdida) desde historico oportunidades
    .join(_hist_opp_mod, on='opportunity_id', how='left')
    # 14. primer booker del comprador
    .join(_primer_booker_df, on='opportunity_id', how='left')
    # 15. flags citas proactivas
    .withColumn('flag_cita_sin_pedido_agendada',
        F.when(F.col('id_am_comprador').cast('long').isin(list(_ids_proactivas)), 1).otherwise(0))
    .withColumn('flag_cita_sin_pedido_show',
        F.when(F.col('id_am_comprador').cast('long').isin(list(_ids_proactivas_show)), 1).otherwise(0))
    # 16. flag caso tomado y credito
    .withColumn('flag_caso_tomado_perf_sc',
        F.when(F.col('fecha_caso_tomado_sc').isNotNull(), 1).otherwise(0))
    .withColumn('flag_perfilamento_credito',
        F.when(F.col('fecha_asignacion_perfilamiento_credito').isNotNull(), 1).otherwise(0))
    # 17. opportunity_created_date_day (solo fecha, sin hora)
    .withColumn('opportunity_created_date_day',
        F.date_format(F.col('opportunity_created_date').cast('date'), 'yyyy-MM-dd'))
)
print(f'oportunidades_consumo (antes de origen): {oportunidades_consumo.count():,}')


In [ ]:
# ---- Calculo de opportunity_source_calculado (origen de credito) ----
# Logica equivalente a _calcular_origen_calculado_credito del pipeline pandas
_ops_objetivo = oportunidades_consumo.filter(
    F.col('opportunity_source').isin(['credito am', 'apartado am'])
).select('opportunity_id', 'opportunity_source', 'fecha_caso_tomado_sc', 'perf_contactado',
         'perf_bc_score', 'perf_comentarios', 'perf_fecha_primer_contacto')

# Fecha de seguridad para simular FECHA_SAFETY = '2100-12-01'
_FECHA_SAFETY = '2100-12-01'

# Para cada simulacion: calcular si fue 'rara' (posterior al caso tomado SC)
_sims_enrich = (
    solicitudes_credito
    .join(_ops_objetivo.select('opportunity_id','fecha_caso_tomado_sc'), on='opportunity_id', how='inner')
    .withColumn('_flag_raro',
        F.when(
            F.col('created_date').cast('timestamp') >
            F.coalesce(F.col('fecha_caso_tomado_sc'), F.lit(_FECHA_SAFETY)).cast('timestamp'),
            1).otherwise(0))
    .withColumn('n_simulaciones',
        F.countDistinct('simulation_name').over(Window.partitionBy('opportunity_id')))
    .withColumn('n_flag_raro',
        F.sum('_flag_raro').over(Window.partitionBy('opportunity_id')))
    # Quitar simulaciones post-caso-tomado (excepto si todas lo son)
    .filter((F.col('_flag_raro') != 1) | (F.col('n_simulaciones') == F.col('n_flag_raro')))
    # Tier de clasificacion: 1=pre-aceptada, 2=rechazada, 3=contingencia
    .withColumn('_tier',
        F.when(F.lower(F.col('status_solicitud')) == 'pre aceptada', 1)
         .when(F.lower(F.col('tipo_conclusion_flujo_credito')) == 'flujo contingente', 3)
         .when(F.lower(F.col('status_solicitud')) == 'rechazada automatica', 2)
         .otherwise(F.lit(None).cast('int')))
)

# Elegir la mejor simulacion por oportunidad
_w_mejor = Window.partitionBy('opportunity_id').orderBy(
    F.col('_tier').asc_nulls_last(), F.col('created_date').desc())
_mejor_sim = (
    _sims_enrich
    .withColumn('_rn', F.row_number().over(_w_mejor))
    .filter(F.col('_rn') == 1)
    .withColumn('opportunity_source_aux',
        F.when(F.lower(F.col('tipo_conclusion_flujo_credito')) == 'flujo contingente',
               F.lit('credito am contingencia'))
         .otherwise(F.lit('credito am api')))
    .withColumn('_suffix',
        F.when(
            (F.col('opportunity_source_aux') == 'credito am api') &
            F.lower(F.coalesce(F.col('status_solicitud'), F.lit(''))).contains('rechaz'),
            F.lit(' rechazado'))
         .when(F.col('opportunity_source_aux') == 'credito am api', F.lit(' aprobado'))
         .otherwise(F.lit('')))
    .withColumn('opportunity_source_calculado',
        F.concat(F.col('opportunity_source_aux'), F.col('_suffix')))
    .select(
        'opportunity_id',
        'opportunity_source_calculado',
        F.col('simulation_name').alias('simulation_name_op'),
        'tipo_credito',
        F.col('created_date').alias('created_date_sim_op'),
    )
)

# Unir origen calculado a oportunidades_consumo
oportunidades_consumo = (
    oportunidades_consumo
    .join(_mejor_sim, on='opportunity_id', how='left')
    .withColumn('opportunity_source_aux',
        F.col('opportunity_source_calculado'))
    .drop('opportunity_source_calculado')
)
print('Origen credito calculado.')


In [ ]:
# ---- Clasificacion PUC para apartados sin simulacion ----
# Equivalente a _calcular_opportunity_source_aux_apartados_sin_simulacion
_score = F.col('perf_bc_score').cast('double')
_puc_filtro = (
    (F.col('opportunity_source') == 'apartado am') &
    (F.lower(F.col('perf_contactado')) == 'si') &
    (F.lower(F.col('perf_interesado')) == 'si') &
    (F.lower(F.coalesce(F.col('perf_intencion_pago'), F.lit(''))).isin(['crédito', 'credito'])) &
    F.col('perf_bc_score').isNotNull() &
    F.col('opportunity_source_aux').isNull()
)
oportunidades_consumo = oportunidades_consumo.withColumn(
    'opportunity_source_aux',
    F.when(_puc_filtro & (_score <= 569), F.lit('apartado puc no apto'))
     .when(_puc_filtro & _score.between(570, 649), F.lit('apartado puc viable kuna'))
     .when(_puc_filtro & _score.between(650, 699), F.lit('apartado puc viable bbva'))
     .when(_puc_filtro & (_score >= 700), F.lit('apartado puc directo eam'))
     .otherwise(F.col('opportunity_source_aux'))
)

# ---- Dedup, filtro de fecha y dummies, seleccion de columnas ----
_w_dedup = Window.partitionBy('opportunity_id').orderBy(
    F.col('fecha_asignacion_perfilamiento_credito').desc_nulls_last())
oportunidades_consumo = (
    oportunidades_consumo
    .withColumn('_rn', F.row_number().over(_w_dedup))
    .filter(F.col('_rn') == 1).drop('_rn')
    .filter(F.col('opportunity_created_date_day') >= FECHA_CORTE)
    .filter(~F.col('id_am_comprador').cast('long').isin(ID_AM_DUMMIES))
)
# Seleccionar solo columnas de salida que existan
_cols_out = [c for c in REPORTE_OPORTUNIDADES_COLS if c in oportunidades_consumo.columns]
oportunidades_consumo = oportunidades_consumo.select(_cols_out)
print(f'oportunidades_consumo FINAL: {oportunidades_consumo.count():,} filas, {len(_cols_out)} cols')


## 4. Reporte Histórico Oportunidades

In [ ]:
# Reporte historico de oportunidades: cambios de stage + fecha de creacion
reporte_historico_oportunidades = (
    historico_oportunidades
    .filter(F.lower(F.trim(F.col('field'))) == 'stagename')
    .withColumn('created_by', F.upper(F.col('created_by')))
    .select('opportunity_id', 'old_value', 'new_value', 'created_by', 'created_date')
    .join(
        oportunidades.select('opportunity_id', 'opportunity_name'),
        on='opportunity_id', how='left'
    )
)
print(f'reporte_historico_oportunidades: {reporte_historico_oportunidades.count():,} filas')


## 5. Reporte Citas

In [ ]:
# ---- Preparacion base de citas ----
# Renombres equivalentes a proc_reporte_citas
citas_cons = (
    citas
    .withColumnRenamed('territorio_cita', 'espacio_cita')
    .withColumnRenamed('fecha_checkin',   'check_in')
    .withColumnRenamed('fecha_checkout',  'check_out')
    # created_time = timestamp original; created_date = solo fecha (string yyyy-MM-dd)
    .withColumnRenamed('created_date', 'created_time')
    .withColumn('created_date',
        F.date_format(F.col('created_time').cast('date'), 'yyyy-MM-dd'))
    .withColumn('sched_date',
        F.date_format(F.col('sched_start_time').cast('date'), 'yyyy-MM-dd'))
    # rol: comprador / vendedor / ''
    .withColumn('rol',
        F.when(F.lower(F.coalesce(F.col('work_type_name'), F.lit(''))).isin(WORK_TYPES_COMPRADOR),
               'comprador')
         .when(F.lower(F.coalesce(F.col('work_type_name'), F.lit(''))).isin(WORK_TYPES_VENDEDOR),
               'vendedor')
         .otherwise(''))
)

# Join clientes: nombre (nickname), email, telefono (phone)
citas_cons = (
    citas_cons
    .join(
        clientes.select(
            'id_am',
            F.col('nickname').alias('nombre'),
            F.col('email'),
            F.col('phone').alias('telefono'),
        ),
        on='id_am', how='left'
    )
)

# Join pedidos por commerce_order_id -> sf_order_id
citas_cons = (
    citas_cons
    .join(
        pedidos.select('commerce_order_id', 'sf_order_id').distinct(),
        on='commerce_order_id', how='left'
    )
)

# Join oportunidades_consumo: atributos de la oportunidad
_opps_attrs = oportunidades_consumo.select(
    'opportunity_id',
    F.col('owner_id').alias('opportunity_owner_id'),
    'opportunity_owner', 'opportunity_owner_equipo',
    'perf_bc_score', 'perf_intencion_pago', 'opportunity_stage',
    'opportunity_created_date', 'fecha_asignacion',
    'id_am_comprador', 'opportunity_source', 'opportunity_source_aux',
    'opportunity_name',
).distinct()
citas_cons = citas_cons.join(_opps_attrs, on='opportunity_id', how='left')
print(f'citas_cons base: {citas_cons.count():,}')


In [ ]:
# ---- Booker: quien agendo la cita (status null -> 'scheduled' en historico_citas) ----
_w_bk = Window.partitionBy('numero_cita').orderBy('created_date')
_booker_scheduled = (
    historico_citas
    .filter(F.lower(F.trim(F.col('field')))     == 'status')
    .filter(F.col('old_value').isNull())
    .filter(F.lower(F.trim(F.col('new_value'))) == 'scheduled')
    .withColumn('_rn', F.row_number().over(_w_bk))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        'numero_cita',
        F.col('created_by_id').alias('booker_id'),
        F.upper(F.col('created_by')).alias('booker_name'),
    )
)

# Fallback: si no hay evento scheduled, usar el evento 'created'
_w_cr = Window.partitionBy('numero_cita').orderBy('created_date')
_booker_created = (
    historico_citas
    .filter(F.lower(F.trim(F.col('field'))) == 'created')
    .filter(F.col('created_by').isNotNull())
    .withColumn('_rn', F.row_number().over(_w_cr))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        'numero_cita',
        F.col('created_by_id').alias('hcita_created_by_id'),
        F.upper(F.col('created_by')).alias('hcita_created_by'),
    )
)

citas_cons = (
    citas_cons
    .join(_booker_scheduled, on='numero_cita', how='left')
    .join(_booker_created,   on='numero_cita', how='left')
    .withColumn('flag_booker_origen_nulo',
        F.when(F.col('booker_id').isNull(), 1).otherwise(0))
    .withColumn('booker_id',
        F.coalesce(F.col('booker_id'), F.col('hcita_created_by_id')))
    .withColumn('booker_name',
        F.coalesce(F.col('booker_name'), F.col('hcita_created_by')))
    .drop('hcita_created_by_id', 'hcita_created_by')
)

# SCD2: equipo del booker y equipo del opportunity_owner
citas_cons = add_team_scd2(citas_cons, 'created_date',           'booker_id',           'booker_equipo')
citas_cons = add_team_scd2(citas_cons, 'opportunity_created_date','opportunity_owner_id','opportunity_owner_equipo')
print(f'citas_cons con booker: {citas_cons.count():,}')


In [ ]:
# ---- Flag histshow: status -> completed/in progress en historico ----
_w_hs = Window.partitionBy('numero_cita').orderBy('created_date')
_histshow = (
    historico_citas
    .filter(F.lower(F.trim(F.col('field'))) == 'status')
    .filter(F.lower(F.trim(F.col('new_value'))).isin(STATUS_HISTSHOW))
    .withColumn('_rn', F.row_number().over(_w_hs))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        'numero_cita',
        F.col('old_value').alias('status_old_value'),
        F.col('new_value').alias('status_new_value'),
        F.lit(1).alias('kpi_citas_flag_histshow_compr'),
        F.col('created_date').alias('kpi_citas_fecha_histshow_compr'),
    )
)
citas_cons = (
    citas_cons
    .join(_histshow, on='numero_cita', how='left')
    .withColumn('kpi_citas_flag_histshow_compr',
        F.coalesce(F.col('kpi_citas_flag_histshow_compr'), F.lit(0)))
)

# ---- Cancelaciones: quien cancelo y por que ----
_w_can = Window.partitionBy('numero_cita').orderBy(F.col('created_date').desc())
_cancelaciones = (
    historico_citas
    .filter(F.lower(F.trim(F.col('field'))) == 'status')
    .filter(F.lower(F.trim(F.col('new_value'))).isin(STATUS_CANCELADA))
    .withColumn('_rn', F.row_number().over(_w_can))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        'numero_cita',
        F.col('new_value').alias('status_cancelacion'),
        F.col('created_by_id').alias('cancelado_por_id'),
        F.upper(F.col('created_by')).alias('cancelado_por'),
        F.col('created_date').alias('_cancel_fecha'),
    )
)
citas_cons = (
    citas_cons
    .join(_cancelaciones, on='numero_cita', how='left')
    # Solo mostrar info de cancelacion si la cita esta efectivamente cancelada
    .withColumn('status_cancelacion',
        F.when(F.lower(F.col('status')).rlike('cancelado|no show'),
               F.col('status_cancelacion')))
    .withColumn('cancelado_por_id',
        F.when(F.lower(F.col('status')).rlike('cancelado|no show'),
               F.col('cancelado_por_id')))
    .withColumn('cancelado_por',
        F.when(F.lower(F.col('status')).rlike('cancelado|no show'),
               F.col('cancelado_por')))
)
citas_cons = add_team_scd2(citas_cons, '_cancel_fecha', 'cancelado_por_id', 'cancelado_por_equipo')
citas_cons = (
    citas_cons
    .withColumn('cancelado_por_equipo',
        F.when(F.lower(F.col('status')).rlike('cancelado|no show'),
               F.col('cancelado_por_equipo')))
    .drop('_cancel_fecha')
)
print(f'citas_cons con flags histshow y cancelaciones: {citas_cons.count():,}')


In [ ]:
# ---- Perfilamiento SC owner: quien es el case_owner del caso SC de la oportunidad ----
_perf_sc = (
    casos
    .filter(F.lower(F.trim(F.col('case_subject'))) == 'perfilamiento contact center')
    .select(
        'opportunity_id',
        F.col('case_owner_name_perf_sc').alias('perf_sc_case_owner_name'),
        F.col('case_owner_id_perf_sc').alias('perf_sc_case_owner_id'),
    )
)
citas_cons = citas_cons.join(_perf_sc, on='opportunity_id', how='left')


In [ ]:
# ---- Flags finales y equipo operativo ----
_equipos_operativos_map = (
    {e: 'espacios fisicos' for e in EQUIPOS_ESPACIOS} |
    {e: 'sales center' for e in EQUIPOS_SC} |
    {'integration user': 'integration user'}
)

# Crear columna de mapeo usando CASE WHEN
def map_equipo_operativo(col_name, out_name):
    expr = F.lit('desconocido')
    for equipo, operativo in _equipos_operativos_map.items():
        expr = F.when(F.lower(F.col(col_name)) == equipo, operativo).otherwise(expr)
    return F.coalesce(expr, F.lit('desconocido')).alias(out_name)

citas_cons = (
    citas_cons
    .withColumn('sf_order_id',
        F.coalesce(F.col('sf_order_id'), F.lit(-1)))
    .withColumn('rol',
        F.coalesce(F.when(F.col('rol') == '', F.lit('desconocido')), F.col('rol'), F.lit('desconocido')))
    .withColumn('work_type_name',
        F.coalesce(F.when(F.col('work_type_name') == '', F.lit('desconocido')),
                   F.col('work_type_name'), F.lit('desconocido')))
    .withColumn('booker_name',
        F.coalesce(F.col('booker_name'), F.lit('desconocido')))
    .withColumn('booker_equipo',
        F.coalesce(F.col('booker_equipo'), F.lit('desconocido')))
    .withColumn('id_am',
        F.coalesce(F.col('id_am').cast('long'), F.lit(-1)))
    .withColumn('flag_cita_show_comprador',
        F.when(F.lower(F.col('status')).isin(STATUS_SHOW_CITAS), 1).otherwise(0))
    .withColumn('_flag_agend_tmp',
        F.when(
            F.col('rol').isin(['desconocido', 'comprador']) &
            F.col('work_type_name').isin(['desconocido', 'cita inicial visita comprador']),
            1).otherwise(0))
    .withColumn('flag_dummy',
        F.when(
            (F.col('opportunity_id').isNotNull() & (F.col('flag_booker_origen_nulo') == 1)) |
            (F.col('id_am').isin(ID_AM_DUMMIES) & (F.col('_flag_agend_tmp') == 1)),
            1).otherwise(0))
    .withColumn('flag_cita_agendada_comprador',
        F.when((F.col('_flag_agend_tmp') == 1) & (F.col('flag_dummy') == 0), 1).otherwise(0))
    .withColumn('flag_oppid_origen_nulo',
        F.when(F.col('opportunity_id').isNull(), 1).otherwise(0))
    .withColumn('booker_equipo_operativo', map_equipo_operativo('booker_equipo', 'booker_equipo_operativo'))
    .withColumn('cancelado_por_equipo_operativo', map_equipo_operativo('cancelado_por_equipo', 'cancelado_por_equipo_operativo'))
)

# Flag duplicada: misma combinacion de identidad + fecha + pedido (despues de ordenar)
_w_dup = Window.partitionBy('id_am', 'work_type_name', 'rol', 'sched_date', 'sf_order_id').orderBy(
    F.col('rol'), F.col('flag_dummy'), F.col('sf_order_id').desc(),
    F.col('flag_cita_show_comprador').desc(), F.col('created_date').desc())
citas_cons = (
    citas_cons
    .withColumn('flag_duplicada',
        F.when(F.row_number().over(_w_dup) > 1, 1).otherwise(0))
    .drop('_flag_agend_tmp')
)

# Seleccion de columnas de salida
_cols_citas = [c for c in REPORTE_CITAS_COLS if c in citas_cons.columns]
reporte_citas = citas_cons.select(_cols_citas)
print(f'reporte_citas FINAL: {reporte_citas.count():,} filas, {len(_cols_citas)} cols')


## 6. Reporte Simulaciones

In [ ]:
reporte_simulaciones = (
    solicitudes_credito
    .withColumn('flag_eda',
        F.when(F.coalesce(F.col('opportunity_name'), F.lit(''))
                .startswith('CREDITO EDA'), 1).otherwise(0))
)
print(f'reporte_simulaciones: {reporte_simulaciones.count():,} filas')


## 7. Escritura en Gold S3

In [ ]:
def write_gold(df, nombre):
    path = GOLD_BASE + nombre + '/'
    df.write.mode('overwrite').parquet(path)
    print(f'  Escrito {nombre} -> {path}')

write_gold(oportunidades_consumo,           'ReporteOportunidades')
write_gold(reporte_historico_oportunidades, 'ReporteHistoricoOportunidades')
write_gold(reporte_citas,                   'ReporteCitas')
write_gold(reporte_simulaciones,            'ReporteSimulaciones')


## 8. Fin

In [ ]:
print(f'Pipeline CRM Glue completado: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
